# Qdrant Hybrid Search 실험

- Dense: `intfloat/multilingual-e5-large` (1024d, fastembed)
- Sparse: `Qdrant/bm25` (fastembed)
- Fusion: RRF (Reciprocal Rank Fusion)
- Collection: `products` (3,618개, GP 제외)

In [1]:
import os
import sys
sys.path.insert(0, "../..")

from dotenv import load_dotenv
load_dotenv("../../infra/.env")

from qdrant_client import QdrantClient
from qdrant_client.models import (
    FieldCondition,
    Filter,
    Fusion,
    FusionQuery,
    MatchAny,
    MatchValue,
    Prefetch,
    SparseVector,
)
from fastembed import SparseTextEmbedding, TextEmbedding

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION = "products"

client = QdrantClient(url=QDRANT_URL)
info = client.get_collection(COLLECTION)
print(f"Collection: {COLLECTION}")
print(f"Points: {info.points_count:,}")

/opt/anaconda3/envs/final-project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection: products
Points: 3,618


In [2]:
# 모델 로드 (처음 실행 시 다운로드)
print("Dense 모델 로드 중...")
dense_model = TextEmbedding("intfloat/multilingual-e5-large")
print("Sparse 모델 로드 중...")
sparse_model = SparseTextEmbedding("Qdrant/bm25")
print("완료")

Dense 모델 로드 중...


/var/folders/n6/9fr973913k7931ymsg4ly6rc0000gn/T/ipykernel_26429/374516251.py:3: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  dense_model = TextEmbedding("intfloat/multilingual-e5-large")


Sparse 모델 로드 중...
완료


In [11]:
def embed(query: str):
    """쿼리 → dense / sparse 벡터"""
    prefixed = f"query: {query}"
    dense_vec = list(dense_model.embed([prefixed]))[0].tolist()
    sparse_vec = list(sparse_model.embed([query]))[0]
    return dense_vec, SparseVector(
        indices=sparse_vec.indices.tolist(),
        values=sparse_vec.values.tolist(),
    )


def hybrid_search(
    query: str,
    top_k: int = 10,
    filters: dict | None = None,
):
    """
    Hybrid Search (Dense + Sparse → RRF)

    filters 예시:
      {"pet_type": "강아지"}
      {"category": ["사료", "간식"]}   # any match
      {"sold_out": False}
    """
    dense_vec, sparse_vec = embed(query)

    must = []
    if filters:
        for key, val in filters.items():
            if isinstance(val, list):
                must.append(FieldCondition(key=key, match=MatchAny(any=val)))
            else:
                must.append(FieldCondition(key=key, match=MatchValue(value=val)))
    qdrant_filter = Filter(must=must) if must else None

    results = client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            Prefetch(
                query=dense_vec,
                using="dense",
                limit=top_k * 3,
                filter=qdrant_filter,
            ),
            Prefetch(
                query=sparse_vec,
                using="sparse",
                limit=top_k * 3,
                filter=qdrant_filter,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )
    return results.points


def print_results(points, query: str):
    print(f"\n쿼리: '{query}' — {len(points)}건")
    print("-" * 60)
    for i, p in enumerate(points, 1):
        pl = p.payload
        print(
            f"{i:2}. [{pl.get('goods_id')}] {pl.get('product_name', '-')}"
        )
        print(
            f"    브랜드: {pl.get('brand_name', '-')} "
            f"| {pl.get('pet_type')} / {pl.get('category')} / {pl.get('subcategory')}"
        )
        print(
            f"    가격: {pl.get('price'):,}원 "
            f"| 인기점수: {pl.get('popularity_score'):.2f} "
            f"| 감성: {pl.get('sentiment_avg')} "
            f"| 재구매율: {pl.get('repeat_rate')}"
        )
        if pl.get('health_concern_tags'):
            print(f"    건강태그: {pl.get('health_concern_tags')}")
        print(f"    score: {p.score:.4f}")

## 기본 검색

In [12]:
results = hybrid_search("강아지 관절에 좋은 사료")
print_results(results, "강아지 관절에 좋은 사료")


쿼리: '강아지 관절에 좋은 사료' — 10건
------------------------------------------------------------
 1. [PI000005774] 웰츠 독 관절 케어 6kg
    브랜드: 웰츠 | ['강아지'] / ['사료'] / ['관절']
    가격: 74,000원 | 인기점수: 7.88 | 감성: 0.6018 | 재구매율: 0.0
    건강태그: ['관절']
    score: 0.5000
 2. [GI251123809] 한끼뚝딱 강아지 수제 화식 사료 닭고기 80g
    브랜드: 한끼뚝딱 | ['강아지'] / ['사료'] / ['소프트사료', '습식사료', '화식']
    가격: 2,500원 | 인기점수: 0.00 | 감성: None | 재구매율: None
    score: 0.5000
 3. [GI251124404] 한끼뚝딱 강아지 수제 화식 사료 명태 80g
    브랜드: 한끼뚝딱 | ['강아지'] / ['사료'] / ['소프트사료', '습식사료', '화식']
    가격: 2,500원 | 인기점수: 0.00 | 감성: None | 재구매율: None
    score: 0.3333
 4. [PI000001243] 뉴트리나 건강백서 건강한 관절 2kg
    브랜드: 건강백서 | ['강아지'] / ['사료'] / ['관절']
    가격: 25,000원 | 인기점수: 21.15 | 감성: 0.8781 | 재구매율: 0.1101
    건강태그: ['관절']
    score: 0.3333
 5. [PI000001292] 뉴트리나 건강백서 건강한 관절 10.2kg
    브랜드: 건강백서 | ['강아지'] / ['사료'] / ['관절']
    가격: 81,000원 | 인기점수: 22.56 | 감성: 0.9097 | 재구매율: 0.1984
    건강태그: ['관절']
    score: 0.2500
 6. [GI251123810] 한끼뚝딱 강아지 수제 화식 사료 연어 80g
    브랜드: 한끼

In [13]:
results = hybrid_search("고양이 요로 건강 습식")
print_results(results, "고양이 요로 건강 습식")


쿼리: '고양이 요로 건강 습식' — 10건
------------------------------------------------------------
 1. [GI251102558] 브릿 케어 캣 그레인프리 스테럴라이즈드 요로 케어 400g
    브랜드: 브릿 | ['고양이'] / ['사료'] / ['요로기계']
    가격: 10,500원 | 인기점수: 4.94 | 감성: 0.9446 | 재구매율: 0.0
    건강태그: ['요로']
    score: 0.7000
 2. [GI251130394] 닥터뉴토 올라이즈 고양이 습식 간식 한입 쏘옥 미니캔S 치킨 30g 8개입
    브랜드: 닥터뉴토 | ['고양이'] / ['간식'] / ['간식캔']
    가격: 11,700원 | 인기점수: 2.08 | 감성: 0.0089 | 재구매율: 0.0
    score: 0.5000
 3. [GI251102564] 브릿 케어 캣 그레인프리 스테럴라이즈드 요로 케어 2kg
    브랜드: 브릿 | ['고양이'] / ['사료'] / ['요로기계']
    가격: 41,000원 | 인기점수: 5.49 | 감성: 0.8371 | 재구매율: 0.0
    건강태그: ['요로']
    score: 0.3667
 4. [PI000005823] 힐링타임 키티요요 넥카라 블루 S
    브랜드: 힐링타임 | ['고양이'] / ['용품'] / ['건강관리']
    가격: 10,900원 | 인기점수: 8.08 | 감성: 0.6087 | 재구매율: 0.0
    score: 0.3333
 5. [GI251130395] 닥터뉴토 올라이즈 고양이 습식 간식 한입 쏘옥 미니캔S 참치 30g 8개입
    브랜드: 닥터뉴토 | ['고양이'] / ['간식'] / ['간식캔']
    가격: 11,700원 | 인기점수: 0.00 | 감성: None | 재구매율: None
    score: 0.3333
 6. [GI251130349] 닥터도레미 이빨과자 요로 50g
    브랜드: 닥터도

## 필터 적용 검색

In [6]:
# pet_type 필터
results = hybrid_search(
    "소화에 좋은 간식",
    filters={"pet_type": "강아지", "sold_out": False},
)
print_results(results, "소화에 좋은 간식 (강아지, 재고있음)")


쿼리: '소화에 좋은 간식 (강아지, 재고있음)' — 10건
------------------------------------------------------------
 1. [GI251133818] 든든 | ['강아지', '고양이'] / ['간식'] / ['동결/건조간식', '원물/뼈간식']
    가격: 12,000원 | 인기점수: 4.37 | 감성: 0.6505 | 재구매율: 0.0
    score: 0.5000
 2. [GO251132130] 펫테일 | ['강아지', '고양이'] / ['간식'] / ['수제간식', '영양/기능', '져키/스틱', '져키/트릿']
    가격: 10,000원 | 인기점수: 8.96 | 감성: None | 재구매율: None
    score: 0.5000
 3. [GI251132672] 펫토리아 | ['강아지'] / ['용품'] / ['장난감/훈련']
    가격: 10,900원 | 인기점수: 0.00 | 감성: None | 재구매율: None
    score: 0.3333
 4. [GO251133583] 견우재 | ['강아지'] / ['간식'] / ['수제간식']
    가격: 43,000원 | 인기점수: 3.47 | 감성: None | 재구매율: None
    score: 0.3333
 5. [GI251091600] 개이득 | ['강아지'] / ['간식'] / ['사사미']
    가격: 4,500원 | 인기점수: 19.12 | 감성: 0.9386 | 재구매율: 0.4167
    score: 0.2500
 6. [GI251133111] 레토 | ['강아지'] / ['용품'] / ['장난감/훈련']
    가격: 17,900원 | 인기점수: 3.47 | 감성: 0.9928 | 재구매율: 0.0
    score: 0.2500
 7. [GI251091601] 개이득 | ['강아지'] / ['간식'] / ['사사미']
    가격: 4,500원 | 인기점수: 17.46 | 감성: 0.928 | 재구매율: 0.33

In [7]:
# 건강 태그 필터
results = hybrid_search(
    "치아 관리",
    filters={"health_concern_tags": "치아", "pet_type": "강아지"},
)
print_results(results, "치아 관리 (건강태그=치아, 강아지)")


쿼리: '치아 관리 (건강태그=치아, 강아지)' — 10건
------------------------------------------------------------
 1. [GI251100474] 두카메디 | ['강아지'] / ['덴탈관', '용품'] / ['구강관리', '치약']
    가격: 18,000원 | 인기점수: 3.12 | 감성: 0.9919 | 재구매율: 0.0
    건강태그: ['치아']
    score: 0.5833
 2. [GI251126196] 위시본 | ['강아지'] / ['사료'] / ['관절', '구강/치아(덴탈케어)', '눈/눈물', '스트레스완화', '위장/소화', '체중조절', '피부/모질']
    가격: 120,000원 | 인기점수: 0.00 | 감성: None | 재구매율: None
    건강태그: ['관절', '눈물', '소화', '체중', '치아', '피부']
    score: 0.5000
 3. [GI251093490] 보가덴트 | ['강아지'] / ['덴탈관', '용품'] / ['구강관리', '치약']
    가격: 23,000원 | 인기점수: 16.50 | 감성: 0.8547 | 재구매율: 0.1034
    건강태그: ['치아']
    score: 0.3704
 4. [GI251126195] 위시본 | ['강아지'] / ['사료'] / ['관절', '구강/치아(덴탈케어)', '눈/눈물', '스트레스완화', '위장/소화', '체중조절', '피부/모질']
    가격: 120,000원 | 인기점수: 0.00 | 감성: None | 재구매율: None
    건강태그: ['관절', '눈물', '소화', '체중', '치아', '피부']
    score: 0.3333
 5. [GI251126197] 위시본 | ['강아지'] / ['사료'] / ['관절', '구강/치아(덴탈케어)', '눈/눈물', '스트레스완화', '위장/소화', '체중조절', '피부/모질']
    가격: 120,000원 | 인기점수: 0

## Dense only vs Sparse only vs Hybrid 비교

In [9]:
QUERY = "노견 관절 영양제"
TOP_K = 5
dense_vec, sparse_vec = embed(QUERY)

# Dense only
dense_results = client.query_points(
    collection_name=COLLECTION,
    query=dense_vec,
    using="dense",
    limit=TOP_K,
    with_payload=True,
).points

# Sparse only
sparse_results = client.query_points(
    collection_name=COLLECTION,
    query=sparse_vec,
    using="sparse",
    limit=TOP_K,
    with_payload=True,
).points

# Hybrid (RRF)
hybrid_results = hybrid_search(QUERY, top_k=TOP_K)

def show(label, points):
    print(f"\n{'='*10} {label} {'='*10}")
    for i, p in enumerate(points, 1):
        pl = p.payload
        print(f"{i}. [{pl.get('goods_id')}] {pl.get('product_name')} | {pl.get('category')} | score={p.score:.4f}")

show("Dense only", dense_results)
show("Sparse only", sparse_results)
show("Hybrid (RRF)", hybrid_results)


========== Dense only ==========
1. [GI251126873] 닥터리 프리미엄 관절 영양제 30캡슐 | ['용품'] | score=0.8918
2. [PI000005774] 웰츠 독 관절 케어 6kg | ['사료'] | score=0.8912
3. [GI251118187] 니즈펫닥터 우리아이 관절영양제 | ['용품'] | score=0.8896
4. [PI000001243] 뉴트리나 건강백서 건강한 관절 2kg | ['사료'] | score=0.8870
5. [PI000001292] 뉴트리나 건강백서 건강한 관절 10.2kg | ['사료'] | score=0.8863

========== Sparse only ==========
1. [GI251126873] 닥터리 프리미엄 관절 영양제 30캡슐 | ['용품'] | score=24.9400
2. [GI251124611] 닥터벳 하트츄 워킹 관절 통증 영양제 30정 | ['용품'] | score=24.8087
3. [GI251136157] [무료배송] 농심 반려다움 강아지 관절 영양제 조인트 서포트 15개입 | ['용품'] | score=24.6142
4. [GO251129414] 멍템 관절&연골 케어 영양제 120정 | ['간식'] | score=24.2969
5. [GI251129982] 리스펫 강아지 노령견 노견 간식 견양갱 7gx30p | ['간식'] | score=21.4305

========== Hybrid (RRF) ==========
1. [GI251126873] 닥터리 프리미엄 관절 영양제 30캡슐 | ['용품'] | score=1.0000
2. [GI251124611] 닥터벳 하트츄 워킹 관절 통증 영양제 30정 | ['용품'] | score=0.4444
3. [GI251136157] [무료배송] 농심 반려다움 강아지 관절 영양제 조인트 서포트 15개입 | ['용품'] | score=0.3409
4. [PI000005774] 웰츠 독 관절 케어 6kg | ['사료'

## 성분 기반 검색 (ingredient_text_ocr 활용)

In [10]:
# 원재료에 '연어' 포함 상품
results = hybrid_search(
    "연어 단백질 사료",
    top_k=5,
    filters={"pet_type": "고양이"},
)
print_results(results, "연어 단백질 사료 (고양이)")

# OCR 텍스트 확인
for p in results[:3]:
    ocr = p.payload.get("ingredient_text_ocr")
    if ocr:
        print(f"\n[{p.payload['goods_id']}] OCR 일부: {ocr[:200]}")


쿼리: '연어 단백질 사료 (고양이)' — 5건
------------------------------------------------------------
 1. [GI251128528] 내추럴발란스 | ['고양이'] / ['사료'] / ['맛보기샘플']
    가격: 1,000원 | 인기점수: 6.03 | 감성: 0.9787 | 재구매율: 0.0
    score: 0.6000
 2. [GI251126657] 든든 | ['강아지', '고양이'] / ['간식'] / ['동결/건조간식']
    가격: 27,000원 | 인기점수: 11.93 | 감성: 0.7866 | 재구매율: 0.3636
    score: 0.5667
 3. [GS251133327] 케어캣 | ['고양이'] / ['사료'] / ['전연령']
    가격: 37,800원 | 인기점수: 3.47 | 감성: 0.7056 | 재구매율: 0.0
    score: 0.3333
 4. [GI251132392] 캐츠랑 | ['고양이'] / ['간식'] / ['스낵/캔디']
    가격: 3,500원 | 인기점수: 8.05 | 감성: 0.9507 | 재구매율: 0.0
    score: 0.3333
 5. [GI251127642] 레오나르도 | ['고양이'] / ['사료'] / ['시니어(7세이상)']
    가격: 42,900원 | 인기점수: 8.24 | 감성: 0.9398 | 재구매율: 0.2
    score: 0.3214

[GI251128528] OCR 일부: FEED WITH
CONFIDENCE
EVERY BATCH TESTED
FOR QUALITY & SAFETY
OCEAN FIST
ULTRA
NATURAL
BALANCE.
KOREA, INC.
MADE
반려동물 ALL 건강케어를 위한
내추럴 사이언스 솔루션
The Natural Science Solution for Sensitive Pet
균형잡힌 영양과 맛

[GI251126657] OCR 일부: MADE
KOREA
PO
freeze E